## Hyperparameter tuning

For this task we are testing the tool Optuna, which runs several training trials with various options and keeps the best performing model based on the metric of choice. It shall be set to optimize the roc_auc_score

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
import traceback
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

In [5]:
# Set paths and load raw as well as encoded datasets

ROOT = Path.cwd().parent # Path is anchored relative to this notebook location

DATA = ROOT / "data"

train_encoded = pd.read_csv(DATA / "processed" / "training_fe_full.csv", index_col="respondent_id")
test_encoded = pd.read_csv(DATA / "processed" / "test_fe_full.csv", index_col="respondent_id")

train_ft_raw = pd.read_csv(DATA / "raw" / "training_set_features.csv", index_col="respondent_id")
train_lb_raw = pd.read_csv(DATA / "raw" / "training_set_labels.csv", index_col="respondent_id")
test_raw = pd.read_csv(DATA / "raw" / "test_set_features.csv", index_col="respondent_id")

# Merge raw data into one single dataframe
train_raw = train_ft_raw.merge(train_lb_raw, left_index=True, right_index=True)

In [6]:
# Define targets and features
target_cols = ['h1n1_vaccine', 'seasonal_vaccine']
categorical_features = train_ft_raw.select_dtypes(include=['object', 'category']).columns.tolist()

X_raw = train_ft_raw
y1 = train_lb_raw['h1n1_vaccine']
y2 = train_lb_raw['seasonal_vaccine']

In [7]:
# Optuna function

def objective(trial):
    """Objective function for Optuna with built-in error handling."""
    model_name = trial.suggest_categorical("model", ["lightgbm", "catboost"])

    if model_name == "lightgbm":
        params = {
            "n_estimators": trial.suggest_int("lgb_n_estimators", 100, 1000),
            "learning_rate": trial.suggest_float("lgb_learning_rate", 1e-3, 0.3, log=True),
            "num_leaves": trial.suggest_int("lgb_num_leaves", 16, 256),
            "max_depth": trial.suggest_int("lgb_max_depth", 3, 15),
            "subsample": trial.suggest_float("lgb_subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("lgb_colsample_bytree", 0.5, 1.0),
            "random_state": 42
        }

    else:  # CatBoost
        params = {
            "iterations": trial.suggest_int("cb_iterations", 200, 1000),
            "learning_rate": trial.suggest_float("cb_learning_rate", 1e-3, 0.3, log=True),
            "depth": trial.suggest_int("cb_depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("cb_l2_leaf_reg", 1e-3, 10, log=True),
            "random_seed": 42,
            "verbose": 0
        }

    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X_raw, y1 + y2), 1):
        try:
            X_train_raw, X_valid_raw = X_raw.iloc[train_idx], X_raw.iloc[valid_idx]
            y1_train, y1_valid = y1.iloc[train_idx], y1.iloc[valid_idx]
            y2_train, y2_valid = y2.iloc[train_idx], y2.iloc[valid_idx]

            if model_name == "catboost":
                cat_idx = [X_raw.columns.get_loc(c) for c in categorical_features]
                model1 = CatBoostClassifier(**params)
                model2 = CatBoostClassifier(**params)

                model1.fit(X_train_raw, y1_train, cat_features=cat_idx)
                model2.fit(X_train_raw, y2_train, cat_features=cat_idx)

                pred1 = model1.predict_proba(X_valid_raw)[:, 1]
                pred2 = model2.predict_proba(X_valid_raw)[:, 1]

            else:  # LightGBM
                try:
                    # Try pre-encoded version
                    X_encoded = train_encoded.loc[X_raw.index]
                    X_train = X_encoded.iloc[train_idx]
                    X_valid = X_encoded.iloc[valid_idx]
                except Exception as e:
                    if VERBOSE:
                        print(f"⚠️ Fold {fold}: Using on-the-fly one-hot encoding due to {e}")
                    X_train = pd.get_dummies(X_train_raw, drop_first=True)
                    X_valid = pd.get_dummies(X_valid_raw, drop_first=True)
                    X_train, X_valid = X_train.align(X_valid, join='left', axis=1, fill_value=0)

                model1 = lgb.LGBMClassifier(**params)
                model2 = lgb.LGBMClassifier(**params)
                model1.fit(X_train, y1_train)
                model2.fit(X_train, y2_train)

                pred1 = model1.predict_proba(X_valid)[:, 1]
                pred2 = model2.predict_proba(X_valid)[:, 1]

            auc1 = roc_auc_score(y1_valid, pred1)
            auc2 = roc_auc_score(y2_valid, pred2)
            fold_scores.append(np.mean([auc1, auc2]))

            if VERBOSE:
                print(f"✅ {model_name} | Fold {fold}/{N_FOLDS} | AUC1={auc1:.4f} | AUC2={auc2:.4f}")

        except Exception as e:
            print(f"❌ Error in fold {fold}: {e}")
            if VERBOSE:
                traceback.print_exc()
            # If a fold fails completely, penalize the score
            return 0.0

    return np.mean(fold_scores)


In [10]:
# Initial config
VERBOSE = True
N_TRIALS = 30
N_FOLDS = 5

# Initialize study
optimization_study_1 = optuna.create_study(direction="maximize")

# Run Optuna study
with tqdm(total=N_TRIALS, desc="Optuna Trials", ncols=90) as pbar:
    def tqdm_callback(study, trial):
        pbar.update(1)
        if VERBOSE:
            print(f"\nTrial {trial.number} done | Value: {trial.value:.4f} | Params: {trial.params}\n")

    optimization_study_1.optimize(objective, n_trials=N_TRIALS, callbacks=[tqdm_callback])

pbar.close()

print("Best trial:")
print(optimization_study_1.best_trial.params)
print("Best mean CV ROC-AUC:", optimization_study_1.best_value)

[I 2025-11-11 20:32:49,719] A new study created in memory with name: no-name-c65211f0-6487-4560-9ae8-0baa51a1fbca
Optuna Trials:   0%|                                               | 0/30 [00:00<?, ?it/s]Traceback (most recent call last):
  File "_catboost.pyx", line 2600, in _catboost.get_cat_factor_bytes_representation
  File "_catboost.pyx", line 2115, in _catboost.get_id_object_bytes_string_representation
_catboost.CatBoostError: bad object for id: nan

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\blxck\AppData\Local\Temp\ipykernel_30808\1113775441.py", line 42, in objective
    model1.fit(X_train_raw, y1_train, cat_features=cat_idx)
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\catboost\core.py", line 5245, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_model,
  File "c:\Users\blxck\De

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 0 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 668, 'cb_learning_rate': 0.0029101145402525143, 'cb_depth': 9, 'cb_l2_leaf_reg': 1.7456060003553475}

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 1 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 256, 'cb_learning_rate': 0.001761749788488635, 'cb_depth': 9, 'cb_l2_leaf_reg': 1.9701501673746162}

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 2 done | Value: 0.0000 | Params:

Traceback (most recent call last):
  File "C:\Users\blxck\AppData\Local\Temp\ipykernel_30808\1113775441.py", line 63, in objective
    model1.fit(X_train, y1_train)
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 3656, in __init__
    train_set.construct()
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 2590, in construct
    self._lazy_init(
  File "c:\Users\blxck\Desktop\neuro_flushot\.ven

❌ Error in fold 1: Do not support special JSON characters in feature name.

Trial 3 done | Value: 0.0000 | Params: {'model': 'lightgbm', 'lgb_n_estimators': 864, 'lgb_learning_rate': 0.026692748080862465, 'lgb_num_leaves': 49, 'lgb_max_depth': 8, 'lgb_subsample': 0.8602799913853262, 'lgb_colsample_bytree': 0.795613430737493}



Traceback (most recent call last):
  File "C:\Users\blxck\AppData\Local\Temp\ipykernel_30808\1113775441.py", line 63, in objective
    model1.fit(X_train, y1_train)
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 3656, in __init__
    train_set.construct()
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 2590, in construct
    self._lazy_init(
  File "c:\Users\blxck\Desktop\neuro_flushot\.ven

...


Trial 23 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 848, 'cb_learning_rate': 0.0919894233169752, 'cb_depth': 6, 'cb_l2_leaf_reg': 9.994687683155306}

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 24 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 634, 'cb_learning_rate': 0.004321617488709039, 'cb_depth': 10, 'cb_l2_leaf_reg': 0.3970311328710254}

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 25 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 817, 'cb_learning_rate': 0.01043762319018238, 'cb_depth': 5, 'cb_l2_leaf_reg': 1.9500678127550717}



Traceback (most recent call last):
  File "C:\Users\blxck\AppData\Local\Temp\ipykernel_30808\1113775441.py", line 63, in objective
    model1.fit(X_train, y1_train)
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 3656, in __init__
    train_set.construct()
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 2590, in construct
    self._lazy_init(
  File "c:\Users\blxck\Desktop\neuro_flushot\.ven

❌ Error in fold 1: Do not support special JSON characters in feature name.

Trial 26 done | Value: 0.0000 | Params: {'model': 'lightgbm', 'lgb_n_estimators': 107, 'lgb_learning_rate': 0.25164874809442866, 'lgb_num_leaves': 167, 'lgb_max_depth': 3, 'lgb_subsample': 0.991373967402781, 'lgb_colsample_bytree': 0.9659904906231349}

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 27 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 303, 'cb_learning_rate': 0.0017164856336593212, 'cb_depth': 7, 'cb_l2_leaf_reg': 0.7849383068776559}

❌ Error in fold 1: Invalid type for cat_feature[non-default value idx=21,feature_idx=22]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

Trial 28 done | Value: 0.0000 | Params: {'model': 'catboost', 'cb_iterations': 482, 'cb_

Traceback (most recent call last):
  File "C:\Users\blxck\AppData\Local\Temp\ipykernel_30808\1113775441.py", line 63, in objective
    model1.fit(X_train, y1_train)
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 3656, in __init__
    train_set.construct()
  File "c:\Users\blxck\Desktop\neuro_flushot\.venv\Lib\site-packages\lightgbm\basic.py", line 2590, in construct
    self._lazy_init(
  File "c:\Users\blxck\Desktop\neuro_flushot\.ven

❌ Error in fold 1: Do not support special JSON characters in feature name.

Trial 29 done | Value: 0.0000 | Params: {'model': 'lightgbm', 'lgb_n_estimators': 419, 'lgb_learning_rate': 0.0062284912053991994, 'lgb_num_leaves': 119, 'lgb_max_depth': 15, 'lgb_subsample': 0.5113775776094224, 'lgb_colsample_bytree': 0.6853116072029006}

Best trial:
{'model': 'catboost', 'cb_iterations': 668, 'cb_learning_rate': 0.0029101145402525143, 'cb_depth': 9, 'cb_l2_leaf_reg': 1.7456060003553475}
Best mean CV ROC-AUC: 0.0
